# Flight Computer Telemetry Analysis
Load a cleaned CSV (post-checksum verification) and plot key telemetry channels.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# ---- LOAD DATA ----
CSV_PATH = 'march29Launch_cleaned.csv'  

df = pd.read_csv(CSV_PATH)

# time_us was overwritten with time.time() (epoch seconds) in the GUI
# convert to relative seconds from start of flight
df['t'] = df['time_us'] - df['time_us'].iloc[0]

# Barometer outputs Pa — convert to altitude AGL (meters)
P0 = df['barometer_hMSL_m'].iloc[0]  # ground-level pressure at pad
df['baro_alt_m'] = 44330 * (1 - (df['barometer_hMSL_m'] / P0) ** 0.1903)

# ---- TRIM TO LAUNCH WINDOW ----
# Auto-detect: find first time altitude exceeds 10m, start 5s before that
ALT_THRESHOLD = 100  # meters
LEAD_TIME = 5        # seconds before launch to include
launch_idx = df[df['baro_alt_m'] > ALT_THRESHOLD].index
if len(launch_idx) > 0:
    t_launch = df.loc[launch_idx[0], 't']
    df = df[df['t'] >= (t_launch - LEAD_TIME)].reset_index(drop=True)
    df['t'] = df['t'] - df['t'].iloc[0]  # re-zero time
    print(f'Trimmed to launch window (detected at t={t_launch:.1f}s)')

print(f'Loaded {len(df)} rows, duration: {df["t"].iloc[-1]:.1f}s')
print(max(df['baro_alt_m']), max(df['gps_hMSL_m']), max(df['position']))
df.head()

## 1. Altitude

In [ ]:
fig, ax = plt.subplots()
ax.plot(df['t'], df['baro_alt_m'], label='Barometer (AGL)', linewidth=0.8)
ax.plot(df['t'], df['gps_hMSL_m'], label='GPS', linewidth=0.8, alpha=0.7)
ax.plot(df['t'], df['position'], label='KF Position', linewidth=0.8, alpha=0.7)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Altitude (m)')
ax.set_title('Altitude vs Time (Barometer = AGL from pad)')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Kalman Filter — Acceleration, Velocity, Position

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(df['t'], df['acceleration'], linewidth=0.8, color='tab:red')
axes[0].set_ylabel('Accel (m/s²)')
axes[0].set_title('Kalman Filter Estimates')

axes[1].plot(df['t'], df['velocity'], linewidth=0.8, color='tab:blue')
axes[1].set_ylabel('Velocity (m/s)')

axes[2].plot(df['t'], df['position'], linewidth=0.8, color='tab:green')
axes[2].set_ylabel('Position (m)')
axes[2].set_xlabel('Time (s)')

plt.tight_layout()
plt.show()

## 3. Raw Accelerometer (Body Frame)

In [ ]:
fig, ax = plt.subplots()
#ax.plot(df['t'], df['x_acceleration'], label='X', linewidth=0.6)
#ax.plot(df['t'], df['y_acceleration'], label='Y', linewidth=0.6)
ax.plot(df['t'], df['z_acceleration'], label='Z', linewidth=0.6)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Acceleration (m/s²)')
ax.set_title('Raw Accelerometer (Body Frame)')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Gyroscope (Angular Velocity)

In [ ]:
fig, ax = plt.subplots()
ax.plot(df['t'], np.degrees(df['angular_velocity_x']), label='Roll (X)', linewidth=0.6)
ax.plot(df['t'], np.degrees(df['angular_velocity_y']), label='Pitch (Y)', linewidth=0.6)
ax.plot(df['t'], np.degrees(df['angular_velocity_z']), label='Yaw (Z)', linewidth=0.6)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Angular Velocity (°/s)')
ax.set_title('Gyroscope')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Orientation (Quaternion)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Raw quaternion components
for comp in ['w', 'x', 'y', 'z']:
    axes[0].plot(df['t'], df[comp], label=comp, linewidth=0.6)
axes[0].set_ylabel('Quaternion')
axes[0].set_title('Orientation Quaternion')
axes[0].legend()

# Quaternion norm (should be ~1.0 if data is clean)
qnorm = np.sqrt(df['w']**2 + df['x']**2 + df['y']**2 + df['z']**2)
axes[1].plot(df['t'], qnorm, linewidth=0.6, color='tab:purple')
axes[1].set_ylabel('||q||')
axes[1].set_xlabel('Time (s)')
axes[1].set_title('Quaternion Norm (sanity check — should be ≈1.0)')
axes[1].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 6. Euler Angles (from Quaternion)

In [ ]:
# Convert quaternion -> Euler (roll, pitch, yaw) in degrees
def quat_to_euler(w, x, y, z):
    """Quaternion to Euler angles (ZYX convention), returns degrees."""
    # Roll (x-axis)
    sinr_cosp = 2 * (w * x + y * z)
    cosr_cosp = 1 - 2 * (x * x + y * y)
    roll = np.arctan2(sinr_cosp, cosr_cosp)

    # Pitch (y-axis)
    sinp = 2 * (w * y - z * x)
    pitch = np.where(np.abs(sinp) >= 1, np.copysign(np.pi / 2, sinp), np.arcsin(sinp))

    # Yaw (z-axis)
    siny_cosp = 2 * (w * z + x * y)
    cosy_cosp = 1 - 2 * (y * y + z * z)
    yaw = np.arctan2(siny_cosp, cosy_cosp)

    return np.degrees(roll), np.degrees(pitch), np.degrees(yaw)

roll, pitch, yaw = quat_to_euler(df['w'].values, df['x'].values, df['y'].values, df['z'].values)

fig, ax = plt.subplots()
ax.plot(df['t'], roll, label='Roll', linewidth=0.6)
ax.plot(df['t'], pitch, label='Pitch', linewidth=0.6)
ax.plot(df['t'], yaw, label='Yaw', linewidth=0.6)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Angle (°)')
ax.set_title('Euler Angles (from Quaternion)')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Magnetometer

In [ ]:
fig, ax = plt.subplots()
ax.plot(df['t'], df['gauss_x'], label='X', linewidth=0.6)
ax.plot(df['t'], df['gauss_y'], label='Y', linewidth=0.6)
ax.plot(df['t'], df['gauss_z'], label='Z', linewidth=0.6)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Gauss')
ax.set_title('Magnetometer')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Power — Battery & Pyro Voltage

In [ ]:
fig, ax = plt.subplots()
ax.plot(df['t'], df['main_voltage_v'], label='Main Battery', linewidth=0.8)
ax.plot(df['t'], df['pyro_voltage_v'], label='Pyro Battery', linewidth=0.8)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Voltage (V)')
ax.set_title('Battery Voltages')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Temperature

In [ ]:
fig, ax = plt.subplots()
ax.plot(df['t'], df['temperature_c'], linewidth=0.8, color='tab:orange')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Temperature (°C)')
ax.set_title('Onboard Temperature')
plt.tight_layout()
plt.show()

## 10. GPS Ground Track

In [ ]:
# Filter out 0/0 GPS coords (no fix)
gps = df[(df['latitude'] != 0) & (df['longitude'] != 0)].copy()

if len(gps) > 0:
    fig, ax = plt.subplots(figsize=(8, 8))
    sc = ax.scatter(gps['longitude'], gps['latitude'], c=gps['t'], 
                    cmap='viridis', s=3, edgecolors='none')
    ax.plot(gps['longitude'].iloc[0], gps['latitude'].iloc[0], 'g^', markersize=12, label='Launch')
    ax.plot(gps['longitude'].iloc[-1], gps['latitude'].iloc[-1], 'rv', markersize=12, label='Landing')
    ax.set_xlabel('Longitude (°)')
    ax.set_ylabel('Latitude (°)')
    ax.set_title('GPS Ground Track')
    ax.legend()
    ax.set_aspect('equal')
    plt.colorbar(sc, label='Time (s)')
    plt.tight_layout()
    plt.show()
else:
    print('No valid GPS data (all 0,0). Skipping ground track.')

## 11. 3D Flight Path

In [ ]:
if len(gps) > 0:
    # Convert lat/lon to relative meters from launch point
    lat0, lon0 = gps['latitude'].iloc[0], gps['longitude'].iloc[0]
    m_per_deg_lat = 111320
    m_per_deg_lon = 111320 * np.cos(np.radians(lat0))
    
    x_m = (gps['longitude'] - lon0) * m_per_deg_lon
    y_m = (gps['latitude'] - lat0) * m_per_deg_lat
    z_m = gps['baro_alt_m']

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    ax.plot(x_m, y_m, z_m, linewidth=0.8)
    ax.scatter(x_m.iloc[0], y_m.iloc[0], z_m.iloc[0], c='g', s=80, marker='^', label='Launch')
    ax.scatter(x_m.iloc[-1], y_m.iloc[-1], z_m.iloc[-1], c='r', s=80, marker='v', label='Landing')
    ax.set_xlabel('East (m)')
    ax.set_ylabel('North (m)')
    ax.set_zlabel('Altitude AGL (m)')
    ax.set_title('3D Flight Path')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No valid GPS data. Skipping 3D path.')

## 12. Flight State / Status

In [ ]:
fig, ax = plt.subplots()
ax.plot(df['t'], df['status'], drawstyle='steps-post', linewidth=1.0, color='tab:brown')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Status Code')
ax.set_title('Flight State Machine')
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

## 13. World-Frame Acceleration (Quaternion Rotated)
Rotate raw accel from body frame to world frame using the quaternion, then subtract gravity from Z.

In [ ]:
def rotate_vector_by_quat(w, x, y, z, vx, vy, vz):
    """Rotate vector (vx,vy,vz) by quaternion (w,x,y,z). Returns world-frame vector."""
    # Rotation matrix from quaternion
    r00 = 1 - 2*(y*y + z*z)
    r01 = 2*(x*y - w*z)
    r02 = 2*(x*z + w*y)
    r10 = 2*(x*y + w*z)
    r11 = 1 - 2*(x*x + z*z)
    r12 = 2*(y*z - w*x)
    r20 = 2*(x*z - w*y)
    r21 = 2*(y*z + w*x)
    r22 = 1 - 2*(x*x + y*y)
    
    wx = r00*vx + r01*vy + r02*vz
    wy = r10*vx + r11*vy + r12*vz
    wz = r20*vx + r21*vy + r22*vz
    return wx, wy, wz

ax_w, ay_w, az_w = rotate_vector_by_quat(
    df['w'].values, df['x'].values, df['y'].values, df['z'].values,
    df['x_acceleration'].values, df['y_acceleration'].values, df['z_acceleration'].values
)

# Subtract gravity from world-frame Z
GRAVITY = 9.80665
az_w_no_g = az_w - GRAVITY

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(df['t'], ax_w, label='X (East)', linewidth=0.6)
axes[0].plot(df['t'], ay_w, label='Y (North)', linewidth=0.6)
axes[0].plot(df['t'], az_w, label='Z (Up)', linewidth=0.6)
axes[0].set_ylabel('Acceleration (m/s²)')
axes[0].set_title('World-Frame Acceleration (with gravity)')
axes[0].legend()

axes[1].plot(df['t'], ax_w, label='X (East)', linewidth=0.6)
axes[1].plot(df['t'], ay_w, label='Y (North)', linewidth=0.6)
axes[1].plot(df['t'], az_w_no_g, label='Z (Up) − g', linewidth=0.6)
axes[1].set_ylabel('Acceleration (m/s²)')
axes[1].set_xlabel('Time (s)')
axes[1].set_title('World-Frame Acceleration (gravity removed from Z)')
axes[1].legend()

plt.tight_layout()
plt.show()

## Quick Stats

In [ ]:
print(f"Peak barometric altitude:    {df['baro_alt_m'].max():.1f} m AGL")
print(f"Peak GPS altitude:           {df['gps_hMSL_m'].max():.1f} m MSL")
print(f"Peak KF position:            {df['position'].max():.1f} m")
print(f"Peak KF velocity:            {df['velocity'].max():.1f} m/s")
print(f"Peak KF acceleration:        {df['acceleration'].max():.1f} m/s²")
print(f"Max |raw accel|:             {np.sqrt(df['x_acceleration']**2 + df['y_acceleration']**2 + df['z_acceleration']**2).max():.1f} m/s²")
print(f"Max angular rate:            {np.degrees(np.sqrt(df['angular_velocity_x']**2 + df['angular_velocity_y']**2 + df['angular_velocity_z']**2)).max():.1f} °/s")
print(f"Min main voltage:            {df['main_voltage_v'].min():.2f} V")
print(f"Min pyro voltage:            {df['pyro_voltage_v'].min():.2f} V")
print(f"Temp range:                  {df['temperature_c'].min():.1f} – {df['temperature_c'].max():.1f} °C")
print(f"Max satellites:              {df['satellites'].max()}")
print(f"Flight duration:             {df['t'].iloc[-1]:.1f} s")